In [50]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

In [51]:
# changing directory (please change to your own directory)
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [52]:
# keep months of prediction of each model for each region and season
# for short, medium and long lead time
# short lead time: 0-1 month
# medium lead time: 2-3 month
# long lead time: 4-6 month
def keep_months_of_prediction(df, category):
    temp = df.copy().dropna()
    # keep months of prediction of each model for each region and season
    if category == 'short':
        temp = temp.loc[temp['lead_time'] < 2]
    elif category == 'medium':
        temp = temp.loc[(temp['lead_time'] >= 2) & (temp['lead_time'] < 4)]
    elif category == 'long':
        temp = temp.loc[temp['lead_time'] >= 4]
    else:
        pass
        print("Invalid Category, returning all months")
    return temp.dropna()


In [53]:
# import data (potential skill, AN and BN)
potential_skill = pd.read_csv('data/csv/metrics/stat_clean.csv', index_col=False)
potential_skill = potential_skill.drop(columns = ['conditional_bias', 'unconditional_bias', 'skill_score'])
an = pd.read_csv('data/csv/metrics/region_model_df_high.csv', index_col=False)
bn = pd.read_csv('data/csv/metrics/region_model_df_low.csv', index_col=False)

# removing MME from potential skill
potential_skill = potential_skill.loc[potential_skill['model'] != 'MME']

# remaming columns for clarity
an = an.rename(columns = {'agreement': 'an_agreement'})
bn = bn.rename(columns = {'agreement': 'bn_agreement'})

#merging an and bn to work with one dataframe
an_bn = an.merge(bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

In [63]:
# loading in all the csv files in the data/csv folder for future use
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
list_of_files = glob.glob('data/csv/*.csv')

files_path = []

for path in list_of_files:
    path_mod = path.replace('\\', '/')
    files_path.append(path_mod)

for f in files_path:
    df = pd.read_csv(f)
    df['region'] = '_'.join(f.split('/')[-1].split('_')[0:-3])
    dfs_dict[f] = df

merged_csv_data = pd.concat(dfs_dict.values(), ignore_index=True)

### General SMME Model (using all 7 months of metric and averaging them out)

In [54]:
# keeping all months
potential_skill_general = keep_months_of_prediction(potential_skill, 'all').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_general = potential_skill_general.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_general = merged_metrics_general.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > 0.3 or (an_agreement > 0.4 and bn_agreement > 0.4)
merged_metrics_general = merged_metrics_general[(merged_metrics_general['potential_skill'] > 0.3) | 
                                                ((merged_metrics_general['an_agreement'] > 0.4) & (merged_metrics_general['bn_agreement'] > 0.4))]
merged_metrics_general['region_season'] = merged_metrics_general['region'] + " | " + merged_metrics_general['season'] # compile region and season into one column, split by " | "
merged_metrics_general = merged_metrics_general.drop(columns = ['region', 'season'])

Invalid Category, returning all months


In [84]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_general = merged_metrics_general.groupby('region_season')['model'].apply(list).to_dict()
models_general

# separate the region and season into nested keys
SMME_models_general = {}
SMME_models_general['metrics'] = {
    'potential_skill': 0.3,
    'an_agreement': 0.4,
    'bn_agreement': 0.4,
    'operations': 'potential_skill OR (an AND bn)'
}
for region_season, model_list in models_general.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_general:
        SMME_models_general[region] = {}
    SMME_models_general[region][season] = model_list

# create a text file of the SMME models chosen
with open('SMME_models_general.txt', 'w') as f:
    for region, seasons in SMME_models_general.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")  

In [85]:
# calling the SMME models into a new model called SMME_general
SMME_general_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_general['metrics']
for region, seasons in SMME_models_general.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_general_df = pd.concat([SMME_general_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_general
SMME_general_df = SMME_general_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_general_df['model'] = 'SMME_general'

In [86]:
# Saving the SMME_general_df to a csv file
grouped_general = SMME_general_df.groupby('region')
for region, region_df in grouped_general:
    region_name = str(region)
    model = 'SMME_general'
    filename = f'data/csv/{region_name}_{model}_merged_seasonal.csv'
    region_df.to_csv(filename)

### SMME model that only uses short lead metrics (0-1 month lead time)

In [56]:
# keeping short months
potential_skill_short = keep_months_of_prediction(potential_skill, 'short').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_short = potential_skill_short.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_short = merged_metrics_short.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > 0.3 or (an_agreement > 0.4 and bn_agreement > 0.4)
merged_metrics_short = merged_metrics_short[(merged_metrics_short['potential_skill'] > 0.3) | 
                                                ((merged_metrics_short['an_agreement'] > 0.4) & (merged_metrics_short['bn_agreement'] > 0.4))]
merged_metrics_short['region_season'] = merged_metrics_short['region'] + " | " + merged_metrics_short['season'] # compile region and season into one column, split by " | "
merged_metrics_short = merged_metrics_short.drop(columns = ['region', 'season'])


In [57]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_short = merged_metrics_short.groupby('region_season')['model'].apply(list).to_dict()
models_short

# separate the region and season into nested keys
SMME_models_short = {}
SMME_models_short['metrics'] = {
    'potential_skill': 0.3,
    'an_agreement': 0.4,
    'bn_agreement': 0.4,
    'operations': 'potential_skill OR (an AND bn)'
}
for region_season, model_list in models_short.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_short:
        SMME_models_short[region] = {}
    SMME_models_short[region][season] = model_list
print(SMME_models_short)

# create a text file of the SMME models chosen
with open('SMME_models_short.txt', 'w') as f:
    for region, seasons in SMME_models_short.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")  

{'metrics': {'potential_skill': 0.3, 'an_agreement': 0.4, 'bn_agreement': 0.4, 'operations': 'potential_skill OR (an AND bn)'}, 'eastern_east_africa': {'MAM': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP'], 'OND': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP']}, 'eastern_ukraine': {'AMJ': ['CCSM4', 'CanESM5', 'ECMWF', 'GEM5', 'GFDL', 'METEO'], 'DJF': ['CMCC', 'DWD', 'ECMWF'], 'JA': ['CCSM4', 'CMCC', 'CanESM5', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'NASA', 'NCEP']}, 'lake_victoria_basin': {'DJF': ['CCSM4', 'CESM1', 'CanESM5', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA'], 'MAM': ['CMCC', 'CanESM5', 'ECMWF', 'GFDL', 'JMA', 'NASA', 'NCEP'], 'SON': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP']}, 'south_sudan': {'ASO': ['CMCC', 'CanESM5', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NCEP'], 'JAS': ['CESM1', 'CanESM5', 'DWD', 'ECMWF',

In [87]:
# calling the SMME models into a new model called SMME_short
SMME_short_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_short['metrics']
for region, seasons in SMME_models_short.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_short_df = pd.concat([SMME_short_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_short
SMME_short_df = SMME_short_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_short_df['model'] = 'SMME_short'

In [88]:
# Saving the SMME_short_df to a csv file
grouped_short = SMME_short_df.groupby('region')
for region, region_df in grouped_short:
    region_name = str(region)
    model = 'SMME_short'
    filename = f'data/csv/{region_name}_{model}_merged_seasonal.csv'
    region_df.to_csv(filename)

### SMME model that only uses medium lead metrics (2-3 month lead time)

In [58]:
# keeping medium months
potential_skill_medium = keep_months_of_prediction(potential_skill, 'medium').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_medium = potential_skill_medium.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_medium = merged_metrics_medium.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > 0.3 or (an_agreement > 0.4 and bn_agreement > 0.4)
merged_metrics_medium = merged_metrics_medium[(merged_metrics_medium['potential_skill'] > 0.3) | 
                                                ((merged_metrics_medium['an_agreement'] > 0.4) & (merged_metrics_medium['bn_agreement'] > 0.4))]
merged_metrics_medium['region_season'] = merged_metrics_medium['region'] + " | " + merged_metrics_medium['season'] # compile region and season into one column, split by " | "
merged_metrics_medium = merged_metrics_medium.drop(columns = ['region', 'season'])

In [59]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_medium = merged_metrics_medium.groupby('region_season')['model'].apply(list).to_dict()
models_medium

# separate the region and season into nested keys
SMME_models_medium = {}
SMME_models_medium['metrics'] = {
    'potential_skill': 0.3,
    'an_agreement': 0.4,
    'bn_agreement': 0.4,
    'operations': 'potential_skill OR (an AND bn)'
}
for region_season, model_list in models_medium.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_medium:
        SMME_models_medium[region] = {}
    SMME_models_medium[region][season] = model_list
print(SMME_models_medium)

# create a text file of the SMME models chosen
with open('SMME_models_medium.txt', 'w') as f:
    for region, seasons in SMME_models_medium.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")  

{'metrics': {'potential_skill': 0.3, 'an_agreement': 0.4, 'bn_agreement': 0.4, 'operations': 'potential_skill OR (an AND bn)'}, 'eastern_east_africa': {'MAM': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP'], 'OND': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA']}, 'eastern_ukraine': {'DJF': ['NCEP']}, 'lake_victoria_basin': {'DJF': ['CESM1', 'GEM5', 'GFDL', 'JMA', 'NASA'], 'MAM': ['CCSM4', 'DWD', 'NASA', 'NCEP'], 'SON': ['CanESM5', 'ECMWF', 'METEO']}, 'south_sudan': {'ASO': ['CanESM5', 'DWD', 'ECMWF', 'GEM5', 'JMA', 'METEO', 'NCEP'], 'JAS': ['DWD', 'ECMWF', 'JMA', 'METEO'], 'MJJ': ['ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA']}, 'southern_africa': {'DJF': ['CMCC', 'METEO', 'NCEP'], 'FMA': ['CCSM4', 'ECMWF', 'GFDL', 'NASA']}, 'sri_lanka': {'OND': ['CESM1', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP']}, 'west_africa': {'JAS': ['CanESM5', 'GFDL', 'JMA', 'NASA', 'NCEP']

In [89]:
# calling the SMME models into a new model called SMME_medium
SMME_medium_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_medium['metrics']
for region, seasons in SMME_models_medium.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_medium_df = pd.concat([SMME_medium_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_medium
SMME_medium_df = SMME_medium_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_medium_df['model'] = 'SMME_medium'

In [90]:
# Saving the SMME_medium_df to a csv file
grouped_medium = SMME_medium_df.groupby('region')
for region, region_df in grouped_medium:
    region_name = str(region)
    model = 'SMME_medium'
    filename = f'data/csv/{region_name}_{model}_merged_seasonal.csv'
    region_df.to_csv(filename)

### SMME model that only uses long lead metrics (4-6 month lead time)

In [60]:
# keeping long months
potential_skill_long = keep_months_of_prediction(potential_skill, 'long').drop(columns=['lead_time'])

# merging potential skill with an and bn
merged_metrics_long = potential_skill_long.merge(an_bn, on=['region', 'model', 'season', 'month_of_prediction'], how='inner')

# taking the mean of the month of prediction for each region and model and season
merged_metrics_long = merged_metrics_long.groupby(['region', 'model', 'season'])[['potential_skill', 'an_agreement', 'bn_agreement']].mean().reset_index().dropna()

# keep models with potential skill > 0.3 or (an_agreement > 0.4 and bn_agreement > 0.4)
merged_metrics_long = merged_metrics_long[(merged_metrics_long['potential_skill'] > 0.3) | 
                                                ((merged_metrics_long['an_agreement'] > 0.4) & (merged_metrics_long['bn_agreement'] > 0.4))]
merged_metrics_long['region_season'] = merged_metrics_long['region'] + " | " + merged_metrics_long['season'] # compile region and season into one column, split by " | "
merged_metrics_long = merged_metrics_long.drop(columns = ['region', 'season'])

In [61]:
# create a nested dictionary where the keys are the regions, the values are lists of models
models_long = merged_metrics_long.groupby('region_season')['model'].apply(list).to_dict()
models_long

# separate the region and season into nested keys
SMME_models_long = {}
SMME_models_long['metrics'] = {
    'potential_skill': 0.3,
    'an_agreement': 0.4,
    'bn_agreement': 0.4,
    'operations': 'potential_skill OR (an AND bn)'
}
for region_season, model_list in models_long.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models_long:
        SMME_models_long[region] = {}
    SMME_models_long[region][season] = model_list
print(SMME_models_long)

# create a text file of the SMME models chosen
with open('SMME_models_long.txt', 'w') as f:
    for region, seasons in SMME_models_long.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")  

{'metrics': {'potential_skill': 0.3, 'an_agreement': 0.4, 'bn_agreement': 0.4, 'operations': 'potential_skill OR (an AND bn)'}, 'eastern_east_africa': {'MAM': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'GEM5', 'GFDL', 'METEO', 'NASA'], 'OND': ['CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP']}, 'eastern_ukraine': {'AMJ': ['CanESM5', 'ECMWF'], 'JA': ['CMCC']}, 'lake_victoria_basin': {'DJF': ['JMA'], 'MAM': ['CanESM5', 'ECMWF'], 'SON': ['ECMWF', 'GEM5', 'METEO']}, 'south_sudan': {'ASO': ['NCEP'], 'JAS': ['DWD', 'GEM5', 'JMA', 'METEO'], 'MJJ': ['ECMWF', 'NASA']}, 'southern_africa': {'DJF': ['DWD', 'NCEP'], 'FMA': ['DWD', 'GEM5', 'NCEP']}, 'sri_lanka': {'OND': ['ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA']}, 'west_africa': {'JAS': ['GFDL', 'NASA', 'NCEP']}}


In [91]:
# calling the SMME models into a new model called SMME_long
SMME_long_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
del SMME_models_long['metrics']
for region, seasons in SMME_models_long.items():
    for season, models in seasons.items():
        temp = merged_csv_data[(merged_csv_data['model'].isin(models)) & (merged_csv_data['region'] == region) & (merged_csv_data['season'] == season)]
        SMME_long_df = pd.concat([SMME_long_df, temp], ignore_index=True)

# averaging among the models and renaming to SMME_long
SMME_long_df = SMME_long_df[['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'month_of_prediction', 'year_of_prediction', 'realization_year', 'lead_time'])[['predicted_precip', 'precip']].mean().reset_index()
SMME_long_df['model'] = 'SMME_long'

In [92]:
# Saving the SMME_long_df to a csv file
grouped_long = SMME_long_df.groupby('region')
for region, region_df in grouped_long:
    region_name = str(region)
    model = 'SMME_long'
    filename = f'data/csv/{region_name}_{model}_merged_seasonal.csv'
    region_df.to_csv(filename)

#### Creating the visualization of potential skill


In [93]:
# loading in all the csv files, including the SMME files in the data/csv folder

# Initialize an empty dictionary to store DataFrames
dfs_dict_new = {}

list_of_files_new = glob.glob('data/csv/*.csv')
# Loop over all files
for f in list_of_files_new:
    # Generate the DataFrame
    df = pd.read_csv(f, sep=',', header=0, index_col=False)
    # Store the DataFrame in the dictionary with the year as key
    dfs_dict_new[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict_new.values()).reset_index()

# Drop unnecessary columns
final_df = final_df.drop(columns=['year_of_prediction', 'realization_year', 'index'])

# normalize precipitation values
final_df['precip'] = final_df['precip']/30

In [102]:
# Calculating all the statistics for each region and model inluding the SMME models
# groupby the region, model, season and month_of_prediction and calculate the corr between the predicted and actual precipitation for future use
corr = final_df.groupby(['region', 'model', 'season', 'month_of_prediction', 'lead_time'])[['predicted_precip', 'precip']].corr(method = 'spearman').drop(['precip'], axis = 1).reset_index()
corr = corr.drop(corr.index[::2]).drop(columns = ['level_5'])
corr = corr.rename(columns = {'predicted_precip': 'corr'})

# Calculate mean and standard deviation
stat = final_df.groupby(['region', 'model', 'season', 'month_of_prediction', 'lead_time']).agg(['mean', 'std']).reset_index()
stat = stat.drop(columns=["Unnamed: 0"])
stat.columns = ['region', 'model', 'season', 'month_of_prediction', 'lead_time', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']
print(stat)
# Merging stat and spatial_means_corr to get 1 df with all values
stat_clean_final = stat.merge(corr, left_on=['region', 'model', 'season', 'month_of_prediction', 'lead_time'], 
                        right_on=['region', 'model', 'season', 'month_of_prediction', 'lead_time'], how='left').dropna()

# Calculating metrics
stat_clean_final['potential_skill'] = np.square(stat_clean_final['corr'])
stat_clean_final['conditional_bias'] = np.square(stat_clean_final['corr'] - (stat_clean_final['pred_std'] / stat_clean_final['actual_std']))
stat_clean_final['unconditional_bias'] = np.square((stat_clean_final['pred_mean'] - stat_clean_final['actual_mean']) / stat_clean_final['actual_std'])
stat_clean_final['skill_score'] = stat_clean_final['potential_skill'] - stat_clean_final['conditional_bias'] - stat_clean_final['unconditional_bias']

# drop unnecessary columns
stat_clean_final = stat_clean_final.drop(columns=['corr', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std'])
 
# Save the final DataFrame to a CSV file
stat_clean_final.to_csv('data/csv/metrics/stat_clean_final.csv', index=False)

                   region       model season  month_of_prediction  lead_time  \
0     eastern_east_africa       CCSM4    MAM                    1        2.5   
1     eastern_east_africa       CCSM4    MAM                    2        1.5   
2     eastern_east_africa       CCSM4    MAM                    3        0.5   
3     eastern_east_africa       CCSM4    MAM                    9        6.5   
4     eastern_east_africa       CCSM4    MAM                   10        5.5   
...                   ...         ...    ...                  ...        ...   
1673          west_africa  SMME_short    JAS                    3        4.5   
1674          west_africa  SMME_short    JAS                    4        3.5   
1675          west_africa  SMME_short    JAS                    5        2.5   
1676          west_africa  SMME_short    JAS                    6        1.5   
1677          west_africa  SMME_short    JAS                    7        0.5   

      pred_mean  pred_std  actual_mean 

C:\Users\edwin\AppData\Local\Temp\ipykernel_84712\3127019359.py:9: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  stat = stat.drop(columns=["Unnamed: 0"])


In [105]:
# Create a combined column for the region-season pair
stat_clean_final['region_season'] = stat_clean_final['region'] + " | " + stat_clean_final['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='potential_skill')
    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }
    
    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]
    
    # Create the heatmap and force y ticklabels to remain visible
    ax = sns.heatmap(
        d,
        vmin=0, vmax=0.5,
        cmap=sns.color_palette('Reds', 10),
        fmt=".2f",
        linewidths=0.1,
        linecolor='black',
        square=True,
        yticklabels=True  # ensure ticklabels are drawn
    )
    
    # Set tick label properties explicitly
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)
    ax.tick_params(axis='x', rotation=0)
    ax.invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean_final, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Potential Skill by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/potential_skill.png')
plt.close()

### Visualization for Conditional Bias

In [111]:
# Create a combined column for the region-season pair
stat_clean_final['region_season'] = stat_clean_final['region'] + " | " + stat_clean_final['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='conditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True,
                yticklabels=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

    

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean_final, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Conditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/conditional_bias.png')
plt.close()

### Visualization for Unconditional Bias

In [113]:
# Create a combined column for the region-season pair
stat_clean_final['region_season'] = stat_clean_final['region'] + " | " + stat_clean_final['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='unconditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    ax = sns.heatmap(d,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True,
                yticklabels=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

    # Set tick label properties explicitly
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)
    ax.tick_params(axis='x', rotation=0)
    ax.invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean_final, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Unconditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/unconditional_bias.png')
plt.close()